# 04 — ML Preprocessing

**Thesis:** Comparative Analysis of Machine Learning Algorithms for Predicting CVD Risk in a Tunisian Hospital Population

## Objective
This notebook bridges the descriptive pipeline (01-03) and the supervised machine-learning pipeline (05-06). It:
1. Loads and validates the cleaned dataset produced by `02_Data_Cleaning.ipynb`.
2. Performs a target-leakage check on the cleaned data.
3. Defines the 14 predictors and 1 target, and documents the categorical vs. continuous split required for SMOTENC.
4. Executes a single, stratified 80/20 train/test split before any resampling or scaling is learned.
5. Validates the split for class balance (and batch-segment representation, diagnostic only).
6. Documents the preprocessing strategy applied in Notebook 5 (scaling scope, SMOTENC, imputation).
7. Provides a demonstration-only SMOTENC run to produce the required before/after figures.
8. Saves the split artifacts and updated `ml_config.json`.

> `01_Data_Inspection.ipynb` identified an apparent two-cohort structure in the raw file. This is documented as a methodological limitation. A diagnostic-only batch indicator is reconstructed from row position solely to verify that both segments appear in both splits. **It is never used as a predictive feature.**

## 0. Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
TEST_SIZE = 0.20
np.random.seed(RANDOM_SEED)

PROJECT_DIR = Path.cwd().parent
CLEANED_CSV_PATH = PROJECT_DIR / "data" / "cleaned" / "CVD_cleaned.csv"
FIGURES_DIR = PROJECT_DIR / "figures"
ARTIFACTS_DIR = PROJECT_DIR / "ml_artifacts"
SPLITS_DIR = ARTIFACTS_DIR / "splits"
MODELS_DIR = ARTIFACTS_DIR / "models"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 100, "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11,
})
sns.set_style("whitegrid")

print("Cleaned data :", CLEANED_CSV_PATH)
print("Artifacts dir:", ARTIFACTS_DIR)
print("Splits dir   :", SPLITS_DIR)
print("Figures dir  :", FIGURES_DIR)


Cleaned data : /home/claude/ML_HOSPITAL/data/cleaned/CVD_cleaned.csv
Artifacts dir: /home/claude/ML_HOSPITAL/ml_artifacts
Splits dir   : /home/claude/ML_HOSPITAL/ml_artifacts/splits
Figures dir  : /home/claude/ML_HOSPITAL/figures


In [2]:
def save_fig(fig, filename):
    path = FIGURES_DIR / filename
    fig.savefig(path, bbox_inches="tight", dpi=300)
    print(f"Figure saved -> {path}")
    plt.close(fig)


## 1. Load and Validate the Cleaned Dataset

In [3]:
df = pd.read_csv(CLEANED_CSV_PATH, sep=";")

print("Shape:", df.shape)
assert df.shape == (1529, 15), f"Unexpected shape {df.shape}"
assert df.isna().sum().sum() == 0, "Unexpected missing values."
assert df.duplicated().sum() == 0, "Unexpected duplicate rows."
print("Confirmed: 1529 rows x 15 columns, 0 missing values, 0 duplicates.")
df.head()


Shape: (1529, 15)
Confirmed: 1529 rows x 15 columns, 0 missing values, 0 duplicates.


,Sex,Age,Weight (kg),Height (cm),BMI,Total Cholesterol (mg/dL),HDL (mg/dL),Fasting Blood Sugar (mg/dL),Smoking Status,Diabetes Status,Physical Activity Level,Family History of CVD,Systolic BP,Diastolic BP,CVD Risk Level
0,0,32.0,69.1000,171.000,23.6,248.0,78.0,111.0,0,1,0,0,125.0,79.0,1
1,0,55.0,118.7000,169.000,41.6,162.0,50.0,135.0,1,1,2,1,139.0,70.0,2
2,1,46.0,86.6145,183.000,26.9,103.0,73.0,114.0,0,0,2,1,104.0,77.0,1
3,1,44.0,108.3000,175.694,33.4,134.0,46.0,91.0,0,0,2,1,140.0,83.0,1
4,0,32.0,99.5000,186.000,28.8,146.0,64.0,141.0,1,1,2,0,144.0,83.0,1


## 2. Define Feature Groups and Target

In [4]:
TARGET = "CVD Risk Level"
RISK_LABELS = {0: "LOW", 1: "INTERMEDIARY", 2: "HIGH"}

# 9 continuous numerical predictors -- standardized for Logistic Regression only
CONTINUOUS_NUMERIC = [
    "Age", "Weight (kg)", "Height (cm)", "BMI",
    "Total Cholesterol (mg/dL)", "HDL (mg/dL)", "Fasting Blood Sugar (mg/dL)",
    "Systolic BP", "Diastolic BP",
]

# 5 binary/ordinal predictors -- NOT standardized (meaningful integer encoding)
# These are the categorical_features argument for SMOTENC in Notebook 5
BINARY_ORDINAL = [
    "Sex", "Smoking Status", "Diabetes Status",
    "Physical Activity Level", "Family History of CVD",
]

FEATURES = CONTINUOUS_NUMERIC + BINARY_ORDINAL  # 14 features, fixed order

assert TARGET in df.columns
assert set(FEATURES) == set(df.columns) - {TARGET}
assert sorted(df[TARGET].unique().tolist()) == [0, 1, 2]

suspicious = [c for c in df.columns if "score" in c.lower()]
assert suspicious == [], f"Suspicious columns found: {suspicious}"

print(f"Target   : {TARGET} (0=LOW, 1=INTERMEDIARY, 2=HIGH)")
print(f"Features : {len(FEATURES)}")
print(f"  Continuous: {CONTINUOUS_NUMERIC}")
print(f"  Binary/ord: {BINARY_ORDINAL}")

corr = df[CONTINUOUS_NUMERIC + [TARGET]].corr()[TARGET].drop(TARGET)
print(f"\nMax |r| with target (numerical features): {corr.abs().max():.3f}")
print("(No single numerical feature dominates the target -- no leakage signal.)")


Target   : CVD Risk Level (0=LOW, 1=INTERMEDIARY, 2=HIGH)
Features : 14
  Continuous: ['Age', 'Weight (kg)', 'Height (cm)', 'BMI', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Systolic BP', 'Diastolic BP']
  Binary/ord: ['Sex', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD']

Max |r| with target (numerical features): 0.174
(No single numerical feature dominates the target -- no leakage signal.)


## 3. Target Distribution

In [5]:
y_full = df[TARGET].copy()
target_counts = y_full.value_counts().reindex([0, 1, 2])
target_pct = (y_full.value_counts(normalize=True).reindex([0, 1, 2]) * 100).round(2)

target_summary = pd.DataFrame({
    "Class": [RISK_LABELS[i] for i in [0, 1, 2]],
    "Count": target_counts.values,
    "Percentage (%)": target_pct.values,
})
print("Full dataset class distribution:")
display(target_summary)

fig, ax = plt.subplots(figsize=(6, 5))
palette = {"LOW": "#2ecc71", "INTERMEDIARY": "#f39c12", "HIGH": "#e74c3c"}
bars = ax.bar(target_summary["Class"], target_summary["Count"],
              color=[palette[c] for c in target_summary["Class"]])
for bar, pct in zip(bars, target_summary["Percentage (%)"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{int(bar.get_height())}\n({pct}%)", ha="center", fontsize=10)
ax.set_title("CVD Risk Level -- Full Dataset Class Distribution")
ax.set_xlabel("CVD Risk Level")
ax.set_ylabel("Number of Patients")
save_fig(fig, "01_target_distribution.png")


Full dataset class distribution:


,Class,Count,Percentage (%)
0,LOW,220,14.39
1,INTERMEDIARY,581,38.00
2,HIGH,728,47.61


Figure saved -> /home/claude/ML_HOSPITAL/figures/01_target_distribution.png


## 4. Define X and y; Reconstruct Diagnostic Batch Label

In [6]:
X = df[FEATURES].copy()
y = df[TARGET].copy()

assert len(df) == 1529, "Row count changed - batch diagnostic may be wrong."
_diagnostic_batch = np.where(df.index < 986, "A", "B")
print("X:", X.shape, " y:", y.shape)
print("Diagnostic batch sizes:", dict(zip(*np.unique(_diagnostic_batch, return_counts=True))))


X: (1529, 14)  y: (1529,)
Diagnostic batch sizes: {np.str_('A'): np.int64(986), np.str_('B'): np.int64(543)}


## 5. Stratified 80/20 Train/Test Split

The test set is set aside **once**, before any preprocessing is fit. It will not be touched again until `06_Model_Evaluation.ipynb`. Stratification ensures each split preserves the full-dataset class proportions, which is essential given the minority LOW class.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, batch_train, batch_test = train_test_split(
    X, y, _diagnostic_batch,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y,
)

print(f"Training set : {X_train.shape[0]} rows")
print(f"Test set     : {X_test.shape[0]} rows")
assert X_train.shape[0] == 1223
assert X_test.shape[0] == 306


Training set : 1223 rows
Test set     : 306 rows


## 6. Validate the Split

In [8]:
split_check = pd.DataFrame({
    "Full dataset": y.value_counts(normalize=True).reindex([0,1,2]) * 100,
    "Train": y_train.value_counts(normalize=True).reindex([0,1,2]) * 100,
    "Test": y_test.value_counts(normalize=True).reindex([0,1,2]) * 100,
}).rename(index=RISK_LABELS).round(2)

print("Class distribution (%) across splits:")
display(split_check)

count_check = pd.DataFrame({
    "Train": y_train.value_counts().reindex([0,1,2]),
    "Test" : y_test.value_counts().reindex([0,1,2]),
}).rename(index=RISK_LABELS)
print("\nAbsolute counts:")
display(count_check)


Class distribution (%) across splits:


,Full dataset,Train,Test
CVD Risk Level,,,
LOW,14.39,14.39,14.38
INTERMEDIARY,38.00,38.02,37.91
HIGH,47.61,47.59,47.71



Absolute counts:


,Train,Test
CVD Risk Level,,
LOW,176,44
INTERMEDIARY,465,116
HIGH,582,146


In [9]:
batch_check = pd.DataFrame({
    "Train (%)": pd.Series(batch_train).value_counts(normalize=True) * 100,
    "Test (%)": pd.Series(batch_test).value_counts(normalize=True) * 100,
}).round(1)
print("Diagnostic batch representation (NOT a feature -- for QA only):")
display(batch_check)


Diagnostic batch representation (NOT a feature -- for QA only):


,Train (%),Test (%)
A,65.1,62.1
B,34.9,37.9


## 7. Preprocessing Strategy for Notebook 5

All 14 features are already integer/float-encoded. The preprocessing strategy differs by algorithm:

| Group | Variables | Tree models | Logistic Regression |
|---|---|---|---|
| Continuous (9) | Age, Weight, Height, BMI, Cholesterol, HDL, FBS, SBP, DBP | No scaling | StandardScaler |
| Binary/Ordinal (5) | Sex, Smoking, Diabetes, Activity, Family History | No scaling | No scaling |

A `SimpleImputer(strategy="median")` is included as a safety net in every pipeline (the cleaned dataset has 0 missing values, so it has no effect, but it ensures the pipeline is safe if applied to future data).

### Why SMOTENC instead of SMOTE
Plain SMOTE interpolates between all features using Euclidean distance without distinguishing feature types. Applied to this dataset it would generate fractional values for binary variables (e.g., Sex = 0.37) that are not valid categories. **SMOTENC** handles this correctly: it uses standard KNN interpolation for continuous features and majority-vote among K nearest neighbours for categorical/binary/ordinal features.

In [10]:
assert all(c in X_train.columns for c in BINARY_ORDINAL), \
    "One or more BINARY_ORDINAL columns are missing from X_train."
print("SMOTENC categorical_features argument (column names):")
print(BINARY_ORDINAL)
print("All names confirmed present in X_train.")


SMOTENC categorical_features argument (column names):
['Sex', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD']
All names confirmed present in X_train.


## 8. SMOTENC Demonstration (Training Set Only)

This section demonstrates SMOTENC applied once to the full training set to produce the required before/after figures. **This resampled dataset is discarded immediately after the figures are saved.** The actual modeling pipeline in Notebook 5 re-applies SMOTENC fresh inside each cross-validation fold, as required for a leakage-free pipeline.

In [11]:
from imblearn.over_sampling import SMOTENC

before_counts = y_train.value_counts().reindex([0, 1, 2])
print("Class distribution BEFORE SMOTENC (training set):")
print(before_counts.rename(index=RISK_LABELS))


Class distribution BEFORE SMOTENC (training set):
CVD Risk Level
LOW             176
INTERMEDIARY    465
HIGH            582
Name: count, dtype: int64


In [12]:
smotenc_demo = SMOTENC(
    categorical_features=BINARY_ORDINAL,
    random_state=RANDOM_SEED,
    k_neighbors=5,
)
X_train_res_demo, y_train_res_demo = smotenc_demo.fit_resample(X_train, y_train)

after_counts = pd.Series(y_train_res_demo).value_counts().sort_index()
print("Class distribution AFTER SMOTENC (demonstration -- NOT used for training):")
print(after_counts.rename(index=RISK_LABELS))

for col in BINARY_ORDINAL:
    orig_vals = set(X_train[col].unique())
    new_vals = set(X_train_res_demo[col].unique())
    assert new_vals <= orig_vals, f"{col}: invalid synthetic values {new_vals - orig_vals}"
print("\nCategorical validity check: all synthetic values are within original category sets. PASSED.")


Class distribution AFTER SMOTENC (demonstration -- NOT used for training):
CVD Risk Level
LOW             582
INTERMEDIARY    582
HIGH            582
Name: count, dtype: int64

Categorical validity check: all synthetic values are within original category sets. PASSED.


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)
for ax, counts, title in [
    (axes[0], before_counts, "Before SMOTENC\n(Training Set)"),
    (axes[1], after_counts,  "After SMOTENC\n(Demonstration - not used for training)"),
]:
    labels = [RISK_LABELS[i] for i in [0, 1, 2]]
    bars = ax.bar(labels, counts.values, color=["#2ecc71", "#f39c12", "#e74c3c"])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(int(bar.get_height())), ha="center", fontsize=9)
    ax.set_title(title)
    ax.set_xlabel("CVD Risk Level")
    ax.set_ylabel("Samples")
fig.tight_layout()
save_fig(fig, "02_03_class_distribution_before_after_smote.png")

del X_train_res_demo, y_train_res_demo
print("Demonstration resampled data discarded. Official training data is unchanged.")


Figure saved -> /home/claude/ML_HOSPITAL/figures/02_03_class_distribution_before_after_smote.png
Demonstration resampled data discarded. Official training data is unchanged.


## 9. Save Split Artifacts

In [14]:
X_train.to_csv(SPLITS_DIR / "X_train.csv")
X_test.to_csv(SPLITS_DIR  / "X_test.csv")
y_train.to_csv(SPLITS_DIR / "y_train.csv", header=True)
y_test.to_csv(SPLITS_DIR  / "y_test.csv",  header=True)

print("Saved:")
for name in ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]:
    p = SPLITS_DIR / name
    print(f"  {p}  (exists: {p.exists()})")


Saved:
  /home/claude/ML_HOSPITAL/ml_artifacts/splits/X_train.csv  (exists: True)
  /home/claude/ML_HOSPITAL/ml_artifacts/splits/X_test.csv  (exists: True)
  /home/claude/ML_HOSPITAL/ml_artifacts/splits/y_train.csv  (exists: True)
  /home/claude/ML_HOSPITAL/ml_artifacts/splits/y_test.csv  (exists: True)


## 10. Save Configuration

In [15]:
config = {
    "target": TARGET,
    "risk_labels": {str(k): v for k, v in RISK_LABELS.items()},
    "features": FEATURES,
    "continuous_numeric": CONTINUOUS_NUMERIC,
    "binary_ordinal": BINARY_ORDINAL,
    "random_seed": RANDOM_SEED,
    "test_size": TEST_SIZE,
    "n_train": int(X_train.shape[0]),
    "n_test": int(X_test.shape[0]),
    "smote_method": "SMOTENC",
    "smote_categorical_features": BINARY_ORDINAL,
}

config_path = ARTIFACTS_DIR / "ml_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Configuration saved ->", config_path)
print(json.dumps(config, indent=2))


Configuration saved -> /home/claude/ML_HOSPITAL/ml_artifacts/ml_config.json
{
  "target": "CVD Risk Level",
  "risk_labels": {
    "0": "LOW",
    "1": "INTERMEDIARY",
    "2": "HIGH"
  },
  "features": [
    "Age",
    "Weight (kg)",
    "Height (cm)",
    "BMI",
    "Total Cholesterol (mg/dL)",
    "HDL (mg/dL)",
    "Fasting Blood Sugar (mg/dL)",
    "Systolic BP",
    "Diastolic BP",
    "Sex",
    "Smoking Status",
    "Diabetes Status",
    "Physical Activity Level",
    "Family History of CVD"
  ],
  "continuous_numeric": [
    "Age",
    "Weight (kg)",
    "Height (cm)",
    "BMI",
    "Total Cholesterol (mg/dL)",
    "HDL (mg/dL)",
    "Fasting Blood Sugar (mg/dL)",
    "Systolic BP",
    "Diastolic BP"
  ],
  "binary_ordinal": [
    "Sex",
    "Smoking Status",
    "Diabetes Status",
    "Physical Activity Level",
    "Family History of CVD"
  ],
  "random_seed": 42,
  "test_size": 0.2,
  "n_train": 1223,
  "n_test": 306,
  "smote_method": "SMOTENC",
  "smote_categorical_feat

## 11. Summary

| Step | Result |
|---|---|
| Dataset | 1,529 rows x 15 columns, 0 missing, 0 duplicates |
| Features | 14 (9 continuous + 5 binary/ordinal) |
| Target | CVD Risk Level: LOW=220 (14.4%), INTERMEDIARY=581 (38.0%), HIGH=728 (47.6%) |
| Train size | 1,223 rows |
| Test size | 306 rows |
| Stratification | Class proportions preserved in both splits |
| Leakage check | No suspicious columns; max |r| with target < 0.2 |
| Resampling | SMOTENC (demonstration only here; applied inside CV folds in Notebook 5) |

**What comes next:** `05_Model_Training.ipynb` loads the saved artifacts and trains all four models using leakage-safe `imblearn.Pipeline` (imputation -> optional scaling -> SMOTENC -> classifier) with 5-fold stratified cross-validation and `f1_macro` tuning.